# Pro-bono: Student Reading Campaign Proposal (Regenerated)
source

This section documents how the LLM judge is used, decision thresholds, fallbacks, and how librarians intervene.

- Primary judge: Hermes3 (python client preferred, HTTP /v1/completions fallback).
- Judge output contract: the judge should emit a JSON object with fields: {"score": float (0.0-1.0), "reason": string}. We instruct Hermes3 to put this JSON on the FIRST LINE of its response and provide an explicit fallback JSON when it cannot produce structured JSON.
- Thresholds: score >= 0.85 → approve; score >= 0.45 → review; else reject.
- Persistence: all judge outputs are written to `judge_logs` for audit and to support librarian override. Raw unparseable Hermes HTTP bodies are persisted to `hermes_raw_responses` for diagnostics.
- Human-in-loop: Admin UI surfaces the latest judge label and score and includes Approve/Reject override controls. Auto-approval is gated by `AUTO_APPROVE_THRESHOLD` env var.

### Accessing judge logs and raw Hermes responses

The prototype stores audit and judge data in the SQLite database at `agentic-rag-mvp/data/agent.db` (use Postgres in production). The main tables you will inspect are `judge_logs` and `hermes_raw_responses`.

sqlite3 CLI examples:

```bash
# Show recent judge entries (id, audit_id, model, score, label, snippet of reason, created_at)
sqlite3 agentic-rag-mvp/data/agent.db "SELECT id, audit_id, model, score, label, substr(reason,1,300), created_at FROM judge_logs ORDER BY created_at DESC LIMIT 50;"

# Show recent raw Hermes HTTP bodies captured for diagnostics
sqlite3 agentic-rag-mvp/data/agent.db "SELECT id, audit_id, url, substr(raw_body,1,500), created_at FROM hermes_raw_responses ORDER BY created_at DESC LIMIT 50;"
```

Python snippet (for richer inspection):

```python
import sqlite3, json
con = sqlite3.connect('agentic-rag-mvp/data/agent.db')
cur = con.cursor()
cur.execute("SELECT id, audit_id, model, score, label, reason, created_at FROM judge_logs ORDER BY created_at DESC LIMIT 20")
for r in cur.fetchall():
    print('JUDGE', r[0], r[2], r[3], r[4])

cur.execute("SELECT id, audit_id, url, raw_body, created_at FROM hermes_raw_responses ORDER BY created_at DESC LIMIT 10")
for r in cur.fetchall():
    print('RAW', r[0], r[1], str(r[3])[:300])
con.close()
```

Admin UI helpers (Python) — if you prefer to use the repo helpers:

```python
from agentic_rag_mvp.tools import admin_ui
# List pending audit rows with judge metadata
rows = admin_ui.get_pending_rows()
# Fetch full judge JSON for a given audit row id (replace <audit_id> with an integer)
full = admin_ui.fetch_full_judge_json(<audit_id>)
print(full)
```

Notes:
- `judge_logs.reason` contains the parsed JSON or explanatory text returned by the judge. For audit traceability, keep `judge_logs` append-only and avoid in-place edits.
- `hermes_raw_responses.raw_body` preserves the last raw HTTP response body when parsing failed — collect multiple samples before making server-side changes.
- For production, run the same queries against Postgres (adjust connection and tooling) and secure access to logs (RBAC).

```mermaid
flowchart LR
  subgraph UI[User Interfaces]
    Student["Student Gradio UI (borrow)"]
    Campaign["Campaign Gradio UI"]
    Admin["Admin UI (librarian)"]
  end
  Catalog["Catalog (CSV/DB)"] -->|ingest| Preprocess[Preprocessing & TF-IDF/Embedding build]
  Preprocess --> Embedding[Embedding (sentence-transformers or random mock)]
  Embedding --> VectorStore[Qdrant / managed vector DB]
  VectorStore --> API[FastAPI Recommendation API]
  API --> Campaign
  API --> Student
  Student -->|borrow or checkout| AgentDB[(SQLite / Postgres agent.db audit_logs)]
  Campaign -->|campaign events| AgentDB
  Admin -->|approve/override| AgentDB
  AgentDB -->|pending rows| Judge[LLM Judge (Hermes3 or heuristic)]
  Judge -->|judge_logs| AgentDB
  Judge -->|human review triggers| Admin
  style UI fill:#f9f,stroke:#333,stroke-width:1px
```

## 4) Component descriptions & data model

Example `audit_logs` schema (SQLite / Postgres):
```json
  {"id": integer primary key,"created_at": timestamp,"student_id": text,"action_type": text,"payload": text,"status": text,"notes": text }
```

Example `judge_logs` schema: id, audit_id, model, prompt, score, label, reason, created_at.

## 5) Tools & tech stack
## 6) Hermes3 judge: criteria, fallbacks, and operator flow

This section documents how the LLM judge is used, decision thresholds, fallbacks, and how librarians intervene.

- Primary judge: Hermes3 (python client preferred, HTTP /v1/completions fallback).
- Judge output contract: the judge should emit a JSON object with fields: {"score": float (0.0-1.0), "reason": string}. We instruct Hermes3 to put this JSON on the FIRST LINE of its response and provide an explicit fallback JSON when it cannot produce structured JSON.
- Thresholds: score >= 0.85 → approve; score >= 0.45 → review; else reject.
- Persistence: all judge outputs are written to `judge_logs` for audit and to support librarian override. Raw unparseable Hermes HTTP bodies are persisted to `hermes_raw_responses` for diagnostics.
- Human-in-loop: Admin UI surfaces the latest judge label and score and includes Approve/Reject override controls. Auto-approval is gated by `AUTO_APPROVE_THRESHOLD` env var.

### Accessing judge logs and raw Hermes responses

The prototype stores audit and judge data in the SQLite database at `agentic-rag-mvp/data/agent.db` (use Postgres in production). The main tables you will inspect are `judge_logs` and `hermes_raw_responses`.

sqlite3 CLI examples:

```bash
# Show recent judge entries (id, audit_id, model, score, label, snippet of reason, created_at)
sqlite3 agentic-rag-mvp/data/agent.db "SELECT id, audit_id, model, score, label, substr(reason,1,300), created_at FROM judge_logs ORDER BY created_at DESC LIMIT 50;"

# Show recent raw Hermes HTTP bodies captured for diagnostics
sqlite3 agentic-rag-mvp/data/agent.db "SELECT id, audit_id, url, substr(raw_body,1,500), created_at FROM hermes_raw_responses ORDER BY created_at DESC LIMIT 50;"
```

Python snippet (for richer inspection):

```python
import sqlite3, json
con = sqlite3.connect('agentic-rag-mvp/data/agent.db')
cur = con.cursor()
cur.execute("SELECT id, audit_id, model, score, label, reason, created_at FROM judge_logs ORDER BY created_at DESC LIMIT 20")
for r in cur.fetchall():
    print('JUDGE', r[0], r[2], r[3], r[4])

cur.execute("SELECT id, audit_id, url, raw_body, created_at FROM hermes_raw_responses ORDER BY created_at DESC LIMIT 10")
for r in cur.fetchall():
    print('RAW', r[0], r[1], str(r[3])[:300])
con.close()
```

Admin UI helpers (Python) — if you prefer to use the repo helpers:

```python
from agentic_rag_mvp.tools import admin_ui
# List pending audit rows with judge metadata
rows = admin_ui.get_pending_rows()
# Fetch full judge JSON for a given audit row id (replace <audit_id> with an integer)
full = admin_ui.fetch_full_judge_json(<audit_id>)
print(full)
```

Notes:
- `judge_logs.reason` contains the parsed JSON or explanatory text returned by the judge. For audit traceability, keep `judge_logs` append-only and avoid in-place edits.
- `hermes_raw_responses.raw_body` preserves the last raw HTTP response body when parsing failed — collect multiple samples before making server-side changes.
- For production, run the same queries against Postgres (adjust connection and tooling) and secure access to logs (RBAC).

for r in cur.fetchall():
    print('RAW', r[0], r[1], str(r[3])[:300])
con.close()
```

Admin UI helpers (Python) — if you prefer to use the repo helpers:

```python
from agentic_rag_mvp.tools import admin_ui
# List pending audit rows with judge metadata
rows = admin_ui.get_pending_rows()
# Fetch full judge JSON for a given audit row id
full = admin_ui.fetch_full_judge_json(audit_id)

## 12) API Service: FastAPI endpoints (sketch)

A minimal FastAPI sketch for recommendations; run in a separate process for production.

In [ ]:
# FastAPI sketch (do not run inside notebook in production)
try:
    from fastapi import FastAPI
    from pydantic import BaseModel
    app = FastAPI()
    class RecommendReq(BaseModel):
        student_id: str = None
        context: str = None
    @app.post('/recommend')
    async def recommend(req: RecommendReq):
        recs = recommend_knn(req.context or 'general', top_k=5)
        return {'recommendations': [{'id': r[0], 'title': r[1], 'score': r[2]} for r in recs]}
    print('FastAPI sketch loaded (run with uvicorn in a separate process)')
except Exception as e:
    print('FastAPI not available in this environment:', e)

## 13) Hermes3 judge: prompt tuning & demo (runnable)

This section describes the prompt contract and includes a guarded demo cell that runs the planner, invokes the judge scaffold, and shows Admin UI output. The judge code in the repo (`tools/llm_judge.py`) includes guarded logic to prefer Hermes3 (python client) then HTTP, and falls back to a heuristic.

In [ ]:
# Demo flow: run the agent job to create recommendations, let the judge score them, and show Admin UI output.
# Guarded: won't crash if optional services (Hermes3) are missing.
import subprocess, os, json, sqlite3
ROOT = os.path.abspath(
)
try:
    print('Running agent_job.py to plan and persist recommendations...')
    ret = subprocess.run([os.path.join('.venv','bin','python'), 'agentic-rag-mvp/tools/agent_job.py'], capture_output=True, text=True)
    print('agent_job output (tail):')
    print(ret.stdout[-800:])
    if ret.returncode != 0:
        print('agent_job exited with code', ret.returncode)
        print(ret.stderr[:800])
except Exception as e:
    print('Could not run agent_job.py via subprocess:', e)

# Show the latest pending rows and any associated judge_logs via admin helper
try:
    from agentic_rag_mvp.tools import admin_ui
    rows = admin_ui.get_pending_rows()
    print('Pending audit rows (id,action_type,student_id,book_title,judge_label,judge_score):')
    for r in rows[:20]:
        print(r['id'], r['action_type'], r.get('student_id'), r.get('book_title'), r.get('judge_label'), r.get('judge_score'))
    if rows:
        first = rows[0]['id']
        print('Full judge JSON for first pending id:', first)
        print(admin_ui.fetch_full_judge_json(first) or '(no judge row)')
except Exception as e:
    print('Could not import admin_ui or fetch pending rows:', e)

# Quick DB query as fallback
try:
    DB = os.path.join('agentic-rag-mvp', 'data', 'agent.db')
    if os.path.exists(DB):
        con = sqlite3.connect(DB)
        cur = con.cursor()
        cur.execute("SELECT id, action_type, payload, status FROM audit_logs ORDER BY id DESC LIMIT 10")
        for row in cur.fetchall():
            print('AUDIT:', row[0], row[1], str(row[2])[:120], row[3])
        cur.execute("SELECT id, audit_id, model, score, label, substr(reason,1,140), created_at FROM judge_logs ORDER BY created_at DESC LIMIT 10")
        for row in cur.fetchall():
            print('JUDGE:', row)
        con.close()
    else:
        print('DB not found at', DB)
except Exception as e:
    print('DB query failed:', e)

## 14) Reference architecture diagram (detailed)

Below is a more detailed reference architecture highlighting cloud services and layers.

```mermaid
graph LR
  Browser["Student Browser / Gradio UI"] -->|HTTPS| ALB["Load Balancer / CDN"]
  ALB -->|route| API["FastAPI (Cloud Run / ECS)"]
  API -->|recommend| Vector["Qdrant (managed or self-hosted)"]
  API -->|audit| DB["Postgres (RDS / Cloud SQL)"]
  API -->|enqueue| JudgeQueue["Judge queue (SQS / PubSub)"]
  JudgeQueue -->|worker| JudgeWorker["Judge workers (Fargate / Cloud Run)"]
  JudgeWorker -->|call| Model["Model serving (Hosted LLMs / SageMaker / Vertex / EC2 GPU)"]
  Model -->|response| JudgeWorker --> DB
  Admin["Admin UI"] -->|auth| ALB --> API --> DB
  Storage[S3/GCS] -->|catalog artifacts| API
  Monitoring["Cloud Monitoring / Prometheus"] --> API & Model & Vector & DB
```

## 15) Scaling guidance & checklist

(See the appended cloud guidance for AWS/GCP in the repository docs.)

## 16) Appendix: runbook snippets & commands

- Start Hermes3 (local helper): `agentic-rag-mvp/scripts/run_hermes3.sh`
- Run the planner manually: `.venv/bin/python agentic-rag-mvp/tools/agent_job.py`
- Inspect judge_logs: `sqlite3 agentic-rag-mvp/data/agent.db "SELECT * FROM judge_logs ORDER BY created_at DESC LIMIT 20;"`
- Run the notebook manual test cell to exercise the planner/judge flow (guarded).